# Five-minute bars from your 1-minute bars

Your check of step 1.5: build one trade date's 5-minute bars of MES from the vendor's 1-minute bars in your store, then compare the day's high and low with your charting platform. A trade date's session runs from 17:00 Chicago on the evening before to 16:00 Chicago on it, so compare with the full electronic session on the chart, not the day session.

Set `ROOT` and `TRADE_DATE` in the next cell, pick the `.venv` kernel, and run all cells. The bars come from the front contract, the one that traded the most that day, and prices print in index points.

In [ ]:
import os
from datetime import date

import polars as pl

from fefat.config import resolve_paths
from fefat.market import instruments
from fefat.market.bars import resample_bars
from fefat.market.builtin_calendars import BUILT_IN
from fefat.market.calendar import sessions_of
from fefat.market.layout import PartitionRequest
from fefat.market.schema import Timeframe, TimeUnit
from fefat.market.store import Store

ROOT = "MES"
TRADE_DATE = date(2026, 9, 22)

paths = resolve_paths(os.environ)
store = Store(paths.data_dir)
sessions = sessions_of(BUILT_IN[instruments.get(ROOT).calendar_id], paths.cache_dir)
session = sessions.session(TRADE_DATE)

In [ ]:
minute_partitions = [
    file.partition
    for file in store.list_partitions()
    if file.partition.dataset.root == ROOT and str(file.partition.dataset.schema_name) == "ohlcv-1m"
]
if not minute_partitions:
    raise ValueError(f"Expected 1-minute bars of {ROOT} in {paths.data_dir}, got none.")
request = PartitionRequest(dataset=minute_partitions[0].dataset, start=TRADE_DATE, end=TRADE_DATE)
minute_bars = {
    partition.contract: store.scan_partition(partition)
    .filter(pl.col("trade_date") == TRADE_DATE)
    .collect()
    for partition in minute_partitions
    if request.selects(partition)
}
volumes = {contract: bars.get_column("volume").sum() for contract, bars in minute_bars.items()}
front = max(volumes, key=volumes.__getitem__)
five_minute_bars = resample_bars(
    minute_bars[front],
    Timeframe(count=5, unit=TimeUnit.MINUTE),
    sessions,
    sessions.following(TRADE_DATE).open,
)
print(f"Contracts with bars on {TRADE_DATE}: {volumes}")
print(f"Front contract: {front}, {len(five_minute_bars)} five-minute bars")

## The day's high and low

Compare these with the high and low of the same session on your chart. A print received in the break after the close keeps this trade date in a bar of its own, so the high and low come from the bars opening before the close.

In [ ]:
def chicago(column: str) -> pl.Expr:
    return (
        pl.from_epoch(pl.col(column), time_unit="ns")
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone("America/Chicago")
    )


prices = ("open", "high", "low", "close")
readable = five_minute_bars.filter(pl.col("ts_open") < session.close).select(
    opens=chicago("ts_open"),
    **{price: pl.col(price) / 1_000_000_000 for price in prices},
    volume="volume",
)
high_bar = readable.sort("high", descending=True).row(0, named=True)
low_bar = readable.sort("low").row(0, named=True)
session_times = pl.DataFrame({"at": [session.open, session.close]}).select(chicago("at"))
opens_at, closes_at = session_times.get_column("at").to_list()
print(f"Session of {TRADE_DATE}: {opens_at} to {closes_at} Chicago")
print(f"High {high_bar['high']} in the bar opening {high_bar['opens']}")
print(f"Low {low_bar['low']} in the bar opening {low_bar['opens']}")
readable.head(12)